<a href="https://colab.research.google.com/github/SidharthaSagolsem/doomscroll_classifier/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Doomscroll classifier: train + evaluate on Colab

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA, then compares it to a zero-shot baseline and a TF-IDF baseline.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`
2. Key icon (Secrets) in the left sidebar: add `HF_TOKEN` (a Hugging Face **write** token) and switch **Notebook access** on
3. Set `HF_REPO` in the next cell

Then `Runtime > Run all`. It takes about 15 to 25 minutes. The trained adapter is pushed to Hugging Face, so it survives the Colab session ending (v0 was lost because it only lived in `/content`).

In [6]:
# >>> EDIT THIS <<<
HF_REPO = "SidharthaSgl/doomscroll-qwen2.5-1.5b-lora"

REPO_URL = "https://github.com/SidharthaSagolsem/doomscroll_classifier.git"
assert not HF_REPO.startswith("your-username/"), "Set HF_REPO to <your HF username>/doomscroll-qwen2.5-1.5b-lora"

In [8]:
# 1. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
print(torch.cuda.get_device_name(0))

Tesla T4


In [9]:
# 2. Get the code (main branch)
%cd /content
!rm -rf doomscroll_classifier
!git clone --depth 1 -b main {REPO_URL}
%cd /content/doomscroll_classifier
!git log --oneline -1

/content
Cloning into 'doomscroll_classifier'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 75.80 KiB | 6.89 MiB/s, done.
/content/doomscroll_classifier
ea25bf4 (grafted, HEAD -> main, origin/main, origin/HEAD) Ignore all results*/ folders (may contain real posts)


In [11]:
# 3. Install pinned training libraries (about 2 minutes)
!pip install -q -r requirements-train.txt

In [12]:
# 4. Log in to Hugging Face with the HF_TOKEN secret (never printed)
import os
from google.colab import userdata
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: SidharthaSgl


In [13]:
# 5. Clean + split the data (dedup, audit, near-duplicate groups, stratified split)
!python prepare_data.py
import os
assert os.path.exists("data/processed/test.jsonl"), "prepare_data.py failed - see the error above"

raw rows:            160
malformed dropped:   0
exact dups dropped:  0
near-dup groups:     9 (20 rows)
final rows:          160
split   high-dopamin  brainrot/low       neutral   educational   total
train             34            30            25            24     113
val                7             6             6             5      24
test               7             6             5             5      23
wrote data/processed/{train,val,test}.jsonl and stats.json


In [14]:
!python evaluate.py --model SidharthaSgl/doomscroll-qwen2.5-1.5b-lora --systems finetuned,zero-shot,tfidf --test /content/real_test.jsonl --out-dir results_real
!zip -q -r results_real.zip results_real
from google.colab import files
files.download("results_real.zip")

== finetuned
adapter_config.json: 100% 1.21k/1.21k [00:00<00:00, 461kB/s]
config.json: 100% 660/660 [00:00<00:00, 3.19MB/s]

model.safetensors: downloading bytes:   0% 563k/3.09G [00:00<1:15:50, 678kB/s]
model.safetensors: downloading bytes:   2% 59.6M/3.09G [00:01<00:35, 84.4MB/s, 4.07MB/s  ]
model.safetensors: reconstructing file:   0% 1.18M/3.09G [00:01<41:57, 1.23MB/s, 4.24kB/s  ]
model.safetensors: downloading bytes:   8% 262M/3.09G [00:02<00:12, 231MB/s, 20.7MB/s  ]
model.safetensors: reconstructing file:   0% 4.48M/3.09G [00:02<21:06, 2.43MB/s,  275kB/s  ]
model.safetensors: reconstructing file:   0% 5.45M/3.09G [00:02<16:33, 3.10MB/s,  406kB/s  ]
model.safetensors: reconstructing file:   2% 52.6M/3.09G [00:02<00:52, 57.3MB/s,  493kB/s  ]
model.safetensors: downloading bytes:   9% 289M/3.09G [00:02<00:20, 135MB/s, 25.1MB/s  ]
model.safetensors: downloading bytes:  10% 318M/3.09G [00:02<00:20, 135MB/s, 26.9MB/s  ]
model.safetensors: reconstructing file:   9% 279M/3.09G [00:02<00:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 6. Train (QLoRA) and push the adapter to the Hub
!python train.py --output-dir outputs/lora --push-to-hub --hub-repo {HF_REPO}
import os
assert os.path.exists("outputs/lora/adapter_config.json"), "Training failed - see the error above. Fix it before running the next cells."

train=113 val=24
cuda=True 4bit=True dtype=torch.float16
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
Loading weights: 100% 338/338 [00:03<00:00, 110.93it/s]
Tokenizing train dataset: 100% 113/113 [00:00<00:00, 306.65 examples/s]
Building labels for train dataset: 100% 113/113 [00:00<00:00, 2190.81 examples/s]
Truncating train dataset: 100% 113/113 [00:00<00:00, 4217.33 examples/s]
Dropping fully masked examples from train dataset: 100% 113/113 [00:00<00:00, 11548.08 examples/s]
Tokenizing eval dataset: 100% 24/24 [00:00<00:00, 481.50 examples/s]
Building labels for eval dataset: 100% 24/24 [00:00<00:00, 2654.20 examples/s]
Truncating eval dataset: 100% 24/24 [00:00<00:00, 3711.64 examples/s]
Dropping fully masked examples from eval 

In [ ]:
# 7. Smoke test: the output should be one JSON object, not "Post Post Post..."
!python predict.py --model outputs/lora --text "POV: the rizzler hits the griddy in ohio at 3am fr fr skibidi" --text "Why does bread go stale? Starch molecules recrystallise and push water out, which is called retrogradation."

Loading weights: 100% 338/338 [00:03<00:00, 93.23it/s] 
  generated 2/2
{"text": "POV: the rizzler hits the griddy in ohio at 3am fr fr skibidi", "label": "brainrot/low-attention", "reasoning": "A highly nonsensical, chaotic mashup of internet buzzwords and nonsensical video genres designed purely for rapid consumption.", "valid_json": true}
{"text": "Why does bread go stale? Starch molecules recrystallise and push water out, whic", "label": "educational", "reasoning": "This post provides an objective, scientifically accurate explanation of a common household phenomenon without sensationalizing it.", "valid_json": true}


In [ ]:
# 8. Evaluate: fine-tuned vs zero-shot vs TF-IDF on the held-out test set
!python evaluate.py --model outputs/lora --systems finetuned,zero-shot,tfidf --out-dir results
import os
assert os.path.exists("results/report.md"), "evaluate.py failed - see the error above"

== finetuned
Loading weights: 100% 338/338 [00:02<00:00, 115.06it/s]
  generated 16/23
  generated 23/23
   macro-F1 0.923 (95% CI 0.775-1.000), accuracy 0.913
== zero-shot
Loading weights: 100% 338/338 [00:03<00:00, 108.11it/s]
  generated 16/23
  generated 23/23
   macro-F1 0.616 (95% CI 0.458-0.714), accuracy 0.652
== tfidf
   macro-F1 0.908 (95% CI 0.726-1.000), accuracy 0.913
wrote results/metrics.json and report.md


In [ ]:
# 9. Show the report
from IPython.display import Markdown, display
display(Markdown(open("results/report.md", encoding="utf-8").read()))

# Evaluation report

Test set: `data/processed/test.jsonl` (23 posts). 95% CIs from 1000 bootstrap resamples.

## Summary

| System | Macro-F1 | 95% CI | Accuracy | 95% CI | Valid JSON |
|---|---|---|---|---|---|
| finetuned | 0.923 | 0.775-1.000 | 0.913 | 0.783-1.000 | 100.0% |
| zero-shot | 0.616 | 0.458-0.714 | 0.652 | 0.478-0.826 | 91.3% |
| tfidf | 0.908 | 0.726-1.000 | 0.913 | 0.783-1.000 | n/a |

TF-IDF 5-fold group CV macro-F1 on train+val: 0.925 ± 0.014

## finetuned

| Label | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| high-dopamine ragebait | 1.000 | 0.714 | 0.833 | 7 |
| brainrot/low-attention | 0.750 | 1.000 | 0.857 | 6 |
| neutral | 1.000 | 1.000 | 1.000 | 5 |
| educational | 1.000 | 1.000 | 1.000 | 5 |

Confusion matrix (rows = true, columns = predicted):

| true \ pred | high-dopamine ragebait | brainrot/low-attention | neutral | educational | unparsed |
|---|---|---|---|---|---|
| high-dopamine ragebait | 5 | 2 | 0 | 0 | 0 |
| brainrot/low-attention | 0 | 6 | 0 | 0 | 0 |
| neutral | 0 | 0 | 5 | 0 | 0 |
| educational | 0 | 0 | 0 | 5 | 0 |

Some mistakes:

- true **high-dopamine ragebait**, predicted **brainrot/low-attention**: Okay so today we are making a giant family-sized lasagna inside our clean kitchen sink! First, you just dump five pounds of raw ground beef right down there, th
- true **high-dopamine ragebait**, predicted **brainrot/low-attention**: I am officially DONE with tipping culture in 2024. I literally went to a self-checkout kiosk today, scanned my own energy drink, and the machine asked me for a 

## zero-shot

| Label | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| high-dopamine ragebait | 0.545 | 0.857 | 0.667 | 7 |
| brainrot/low-attention | 0.000 | 0.000 | 0.000 | 6 |
| neutral | 0.833 | 1.000 | 0.909 | 5 |
| educational | 1.000 | 0.800 | 0.889 | 5 |

Confusion matrix (rows = true, columns = predicted):

| true \ pred | high-dopamine ragebait | brainrot/low-attention | neutral | educational | unparsed |
|---|---|---|---|---|---|
| high-dopamine ragebait | 6 | 0 | 1 | 0 | 0 |
| brainrot/low-attention | 4 | 0 | 0 | 0 | 2 |
| neutral | 0 | 0 | 5 | 0 | 0 |
| educational | 1 | 0 | 0 | 4 | 0 |

Some mistakes:

- true **brainrot/low-attention**, predicted **high-dopamine ragebait**: me when the toddler starts the industrial-grade screaming ASMR at the function (it's giving siren head vibes no cap) but the iPad is charging and the sensory fr
- true **high-dopamine ragebait**, predicted **neutral**: Okay so today we are making a giant family-sized lasagna inside our clean kitchen sink! First, you just dump five pounds of raw ground beef right down there, th
- true **brainrot/low-attention**, predicted **unparsed**: pov: baby has 1000% rizz in the crib at 3am when he's supposed to be sleeping but he's just staring at the monitor like 👁️👄👁️ built different fr fr skibidi baby
- true **brainrot/low-attention**, predicted **high-dopamine ragebait**: POV: you're a single plastic straw in 2024 trying to survive the paper straw apocalypse but the turtle with 1000 aura walks in 💀 bro really said let me eat that
- true **educational**, predicted **high-dopamine ragebait**: Quick breakdown of why Large Language Models (LLMs) hallucinate: they don't actually 'know' facts like a database. Instead, they use a probabilistic neural netw
- true **brainrot/low-attention**, predicted **high-dopamine ragebait**: when the global warming rain is hitting different and you gotta bust out the mewing face in front of the polar bears 🤫🧏‍♂️ bro thought he was saving the glacier
- true **brainrot/low-attention**, predicted **unparsed**: POV: when the grimace shake rizz hits the skibidi slicer and you gotta lock in for the 1000 degree knife vs giant gummy bear challenge fr fr 💀 level 10 gyatt co
- true **brainrot/low-attention**, predicted **high-dopamine ragebait**: POV: when the professor has negative rizz but you gotta lock in for that 3 AM academic comeback but the brainrot is too strong 💀 bro thinks he's the skibidi dea

## tfidf

| Label | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| high-dopamine ragebait | 1.000 | 0.857 | 0.923 | 7 |
| brainrot/low-attention | 1.000 | 1.000 | 1.000 | 6 |
| neutral | 0.800 | 0.800 | 0.800 | 5 |
| educational | 0.833 | 1.000 | 0.909 | 5 |

Confusion matrix (rows = true, columns = predicted):

| true \ pred | high-dopamine ragebait | brainrot/low-attention | neutral | educational | unparsed |
|---|---|---|---|---|---|
| high-dopamine ragebait | 6 | 0 | 1 | 0 | 0 |
| brainrot/low-attention | 0 | 6 | 0 | 0 | 0 |
| neutral | 0 | 0 | 4 | 1 | 0 |
| educational | 0 | 0 | 0 | 5 | 0 |

Some mistakes:

- true **high-dopamine ragebait**, predicted **neutral**: Okay so today we are making a giant family-sized lasagna inside our clean kitchen sink! First, you just dump five pounds of raw ground beef right down there, th
- true **neutral**, predicted **educational**: So excited to announce that Florence Pugh has officially joined the cast of the upcoming sci-fi thriller 'Echoes of Silence'! Production starts next month in Lo


In [ ]:
# 10. Download results (metrics, predictions, training log; no model weights)
!cp outputs/lora/training_run.json results/
!cp data/processed/stats.json results/data_stats.json
!zip -q -r results.zip results
from google.colab import files
files.download("results.zip")
print(f"Model: https://huggingface.co/{HF_REPO}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model: https://huggingface.co/SidharthaSgl/doomscroll-qwen2.5-1.5b-lora
